# 01. Dimensional Collapse

This notebook reproduces the layer-resolved effective-dimension analysis for the audited Qwen-0.5B archive.
The local `unified_metrics.csv` uses `effective_dim` rather than `effective_dimension`.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

df = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
df.head()
        


In [ ]:
summary = (
    df.groupby(['layer', 'group'])['effective_dim']
    .agg(['mean', 'sem'])
    .reset_index()
)

plt.figure(figsize=(10, 5))
for group, label in [('G1', 'Direct Fail'), ('G2', 'Direct Success'), ('G3', 'CoT Fail'), ('G4', 'CoT Success')]:
    sub = summary[summary['group'] == group]
    plt.plot(sub['layer'], sub['mean'], label=f'{group}: {label}')
    plt.fill_between(sub['layer'], sub['mean'] - sub['sem'], sub['mean'] + sub['sem'], alpha=0.2)

plt.title('Effective Dimension Across Network Depth')
plt.xlabel('Layer')
plt.ylabel('Mean effective dimension')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'repro_01_dimensional_collapse.png', dpi=300)
plt.show()
        


In [ ]:
layer14 = df[df['layer'] == 14]
g4 = layer14[layer14['group'] == 'G4']['effective_dim']
g1 = layer14[layer14['group'] == 'G1']['effective_dim']
print({'G4_mean': float(g4.mean()), 'G1_mean': float(g1.mean()), 'cohens_d': float(cohens_d(g4, g1))})

layer12 = df[df['layer'] == 12]
g3 = layer12[layer12['group'] == 'G3']['effective_dim']
print({'G4_vs_G3_layer12_d': float(cohens_d(layer12[layer12['group'] == 'G4']['effective_dim'], g3))})
        
